In [1]:
import os

os.environ["HF_HOME"] = "D:/AI_Translate_App/models"
os.environ["TRANSFORMERS_CACHE"] = "D:/AI_Translate_App/models"

In [2]:
pip install transformers torch openai-whisper

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
import os

# 🔥 FIX FFmpeg
os.environ["PATH"] += ";C:/Users/vietr/Downloads/ffmpeg-8.1-essentials_build/bin"

import whisper
from transformers import MarianMTModel, MarianTokenizer

# ===== Load model =====
speech_model = whisper.load_model("base")

model_name = "Helsinki-NLP/opus-mt-en-vi"
tokenizer = MarianTokenizer.from_pretrained(
    model_name,
    cache_dir="D:/AI_Translate_App/models"
)

translator = MarianMTModel.from_pretrained(
    model_name,
    cache_dir="D:/AI_Translate_App/models"
)

# ===== Audio =====
audio_path = "D:/AI_Translate_App/audio/audio.mp3"

result = speech_model.transcribe(audio_path)
text = result["text"]

print("Text goc:", text)

# ===== Hàm chia nhỏ text =====
def split_text(text, max_len=200):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_len):
        chunks.append(" ".join(words[i:i+max_len]))
    return chunks

# ===== Dịch từng đoạn =====
chunks = split_text(text)

translated_text = ""

for chunk in chunks:
    inputs = tokenizer(
        chunk,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    
    translated = translator.generate(**inputs)
    
    translated_text += tokenizer.decode(
        translated[0],
        skip_special_tokens=True
    ) + " "

print("Tieng Viet:", translated_text)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Text goc:  Bây giờ là chúc bị bình thường. Như là đã đi khi từ gà một bên sống hỏi hả? Nhà tối hình phố trong một lần đó, cho là mấy mắn cho tôi tìm. Bụi bài đồng hạnh, chú mong mắn nhé, ca hay lột tạng của một tần xấu. Đâu thể vô vầy là mạng bán mà vòng hồng tồn đồng. Và ta thì xin qua nó với một tầm hồng, đổ để máy ngủy cho ta xin chào. Và đặc nổi đêm tinh vào, không tần linh nào muốn ta hoàng hào. Bà các chào vào đồng của cám ứng mào. Tự dù ý nhất, đen ngắn các cô người, chá là ngôn ngữ. Bình pháp giao chân, ra cô tôn tử. Tời tôi nằm xuống sẽ làm đường về lạnh cho cô thông ngữ. Nơi tôi nằm xuống sẽ là cây, sẽ là giải và sẽ là một lắng mò. Và tôi chung chữ sạch con xinh. Bọn cho cái cây, phụng xân trường, mang nước sáng cốt mùi dừng thường. Sao công việc tay trưng lưng tự trân tựa chạm, bọn thử cho là giám bằng nhất cức. Thả nó vào đời tình lòng, vững bức đất, như ta đạp lần nó với hình thường của bàn trơn. Có lúc tìm tròn, lúc thì vụ nằm tại yêu máy nhà. Điều nó có mèo, biêu không b

In [11]:
pip install pygame gtts

Defaulting to user installation because normal site-packages is not writeable
  Using cached gTTS-2.5.4-py3-none-any.whl.metadata (4.1 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
    --------------------------------------- 0.3/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/10.6 MB 1.2 MB/s eta 0:00:09
   - -------------------------------------- 0.5/10.6 MB 1.2 MB/s eta 0:00:09
   -- ------------------------------------- 0.8/10.6 MB 662.7 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/10.6 MB 662.7 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/10.6 MB 662.7 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/10.6 MB 662.7 kB/s eta 0:00:15
   ---- ----------------------------------- 1.3/10.6 MB 567.2 kB/s eta 0:00:17
   ---- -----------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [12]:
from gtts import gTTS
import pygame

# tạo file mp3
tts = gTTS(translated_text, lang='vi')
tts.save("output.mp3")

# phát âm thanh
pygame.init()
pygame.mixer.init()
pygame.mixer.music.load("output.mp3")
pygame.mixer.music.play()

while pygame.mixer.music.get_busy():
    continue

C:\Users\vietr\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [18]:
from IPython.display import display, Markdown
import ipywidgets as widgets

# ===== UI =====
upload = widgets.FileUpload(accept='.mp3', multiple=False)
btn = widgets.Button(description="Dịch", button_style='success')
output = widgets.Output()

display(upload, btn, output)

def translate_audio(b):
    with output:
        output.clear_output()
        
        if not upload.value:
            print("⚠️ Vui lòng upload file trước!")
            return
        
        file_info = upload.value[0]

        with open("temp.mp3", "wb") as f:
            f.write(file_info['content'])

        print("⏳ Đang xử lý...")

        result = speech_model.transcribe("temp.mp3")
        text = result["text"]

        print("\n📄 Text gốc:")
        print(text)

        # chia nhỏ
        def split_text(text, max_len=200):
            words = text.split()
            return [" ".join(words[i:i+max_len]) for i in range(0, len(words), max_len)]

        chunks = split_text(text)

        translated_text = ""
        for chunk in chunks:
            inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=512)
            translated = translator.generate(**inputs)
            translated_text += tokenizer.decode(translated[0], skip_special_tokens=True) + " "

        print("\n🇻🇳 Bản dịch:")
        print(translated_text)

# ===== Gán sự kiện nút =====
btn.on_click(translate_audio)

FileUpload(value=(), accept='.mp3', description='Upload')

Button(button_style='success', description='Dịch', style=ButtonStyle())

Output()

### Kết quả dịch:

Anh có thể giúp chúng tôi không? Chúng tôi có thể làm gì đây? Xiên thân, Thái Dương Thái Lan, Thái Dương Dương Thái Lan. Em có thể làm gì không? Em hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi hi Tôi có thể giúp gì cho anh ta không? 